Rozwiązanie problemu klasyfikacji wieloklasowej z wykorzystaniem 
sieci neuronowych – Pytorch Lightning, kalibracja, XAI

### Klasyfikacja kategorii bezpieczeństwa metod antykoncepcji na podstawie profilu zdrowotnego kobiet z wykorzystaniem sieci neuronowej, kalibracji prawdopodobieństw oraz metod XAI

MODEL DOSTAJE: stan zdrowia pacjentki + rozważana metoda antykoncepcji

MODEL PRZEWIDUJE: czy ta metoda jest medycznie bezpieczna / ostrożna / niezalecana / przeciwwskazana

Klasy oparte na U.S. Medical Eligibility Criteria for Contraceptive Use 2024, czyli wytycznych CDC. U.S. MEC 2024 zawiera zalecenia dla konkretnych metod antykoncepcji u osób z określonymi chorobami lub cechami klinicznymi.      https://pmc.ncbi.nlm.nih.gov/articles/PMC11315372

Klasy targetu byłyby zgodne z kategoriami MEC:

|     Klasa | Znaczenie                                                                     |
| --------: | ----------------------------------------------------------------------------- |
| **MEC 1** | metoda bez ograniczeń                                                         |
| **MEC 2** | metoda zwykle bezpieczna, korzyści przeważają nad ryzykiem                    |
| **MEC 3** | metoda zwykle niezalecana, chyba że inne opcje są nieakceptowalne/niedostępne |
| **MEC 4** | metoda przeciwwskazana, nieakceptowalne ryzyko                                |


In [21]:
from torch import nn
import lightning.pytorch as pl
from lightning.pytorch import profilers
from lightning.pytorch.loggers import LitLogger
from lightning.fabric import Fabric
from lightning.pytorch import LightningModule, Trainer, seed_everything

In [22]:
logger = LitLogger(name='my_logger')
fabric = Fabric(loggers=logger)

In [23]:
class MNISTClassifier(pl.LightningModule):
    pass        # Pytorch Lightning        

In [24]:
def training_step(self, train_batch, batch_idx):
    x, y = train_batch
    logits = self.forward(x)
    loss = self.cross_entropy_loss(logits, y)
    self.log('train_loss', loss)
    return loss

def validation_step(self, val_batch, batch_idx):
    x,y = val_batch
    logits = self.forward(x)
    loss = self.cross_entropy_loss(logits, y)
    self.log('val_loss', loss)
    return loss

def train_val_one_step(self, batch, batch_idx):
    x,y = batch
    logits = self.forward(x)
    loss = self.cross_entropy_loss(logits, y)
    self.log('loss', loss)
    return loss

In [25]:
profiler = profilers.AdvancedProfiler()
trainer = pl.Trainer(profiler=profiler)
# trainer.fit(LightningMNISTClassifier())

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
import torch
from torch import nn
import pytorch_lightning as pl
from torch.utils.data import DataLoader, random_split
from torch.nn import functional as F
from torchvision.datasets import MNIST
from torchvision import datasets, transforms
import os

class LightningMNISTClassifier(pl.LightningModule):

  def __init__(self):
    super().__init__()

    # mnist images are (1, 28, 28) (channels, width, height) 
    self.layer_1 = torch.nn.Linear(28 * 28, 128)
    self.layer_2 = torch.nn.Linear(128, 256)
    self.layer_3 = torch.nn.Linear(256, 10)

  def forward(self, x):
      batch_size, channels, width, height = x.size()

      # (b, 1, 28, 28) -> (b, 1*28*28)
      x = x.view(batch_size, -1)

      # layer 1 (b, 1*28*28) -> (b, 128)
      x = self.layer_1(x)
      x = torch.relu(x)

      # layer 2 (b, 128) -> (b, 256)
      x = self.layer_2(x)
      x = torch.relu(x)

      # layer 3 (b, 256) -> (b, 10)
      x = self.layer_3(x)

      # probability distribution over labels
      x = torch.log_softmax(x, dim=1)

      return x

  def cross_entropy_loss(self, logits, labels):
    return F.nll_loss(logits, labels)

  def training_step(self, train_batch, batch_idx):
      x, y = train_batch
      logits = self.forward(x)
      loss = self.cross_entropy_loss(logits, y)
      self.log('train_loss', loss)
      return loss


  def validation_step(self, val_batch, batch_idx):
      x, y = val_batch
      logits = self.forward(x)
      loss = self.cross_entropy_loss(logits, y)
      self.log('val_loss', loss)

  def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
    return optimizer


class MNISTDataModule(pl.LightningDataModule):

  def setup(self, stage):
    # transforms for images
    transform=transforms.Compose([transforms.ToTensor(), 
                                  transforms.Normalize((0.1307,), (0.3081,))])
      
    # prepare transforms standard to MNIST
    self.mnist_train = MNIST(os.getcwd(), train=True, download=True, transform=transform)
    self.mnist_test = MNIST(os.getcwd(), train=False, download=True, transform=transform)

  def train_dataloader(self):
    return DataLoader(self.mnist_train, batch_size=64)

  def val_dataloader(self):
    return DataLoader(self.mnist_test, batch_size=64)

data_module = MNISTDataModule()

# train
model = LightningMNISTClassifier()
trainer = pl.Trainer()

trainer.fit(model, data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\ankap\OneDrive\Desktop\PROJEKTY\nn\.venv\Lib\site-packages\pytorch_lightning\loops\utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
100%|██████████| 9.91M/9.91M [00:02<00:00, 4.95MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 242kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.55MB/s]

  | Name    | Type   | Params | Mode  | FLOPs
----------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\ankap\OneDrive\Desktop\PROJEKTY\nn\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\ankap\OneDrive\Desktop\PROJEKTY\nn\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
c:\Users\ankap\OneDrive\Desktop\PROJEKTY\nn\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]